In [ ]:
# scrape_interconnection_projects.py
# ----------------------------------
import json, time, pathlib, traceback
from typing import List, Dict

import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm              # nice progress bar (pip install tqdm)

# ----------------------------------------------------------------------
# 1) CONFIG -------------------------------------------------------------
# ----------------------------------------------------------------------
INPUT_XLSX   = "testdata/Interconnectionfyi_interconnection_queue_data_in_the_US_19962025.2025-8-5.File-2.xlsx"       # <-- your workbook
LINK_COLUMN  = "Project Details"                    # <-- column name that holds URLs
PAUSE_SEC    = 0.05                      # polite delay between requests
OUT_CSV      = "interconnection_projects_2025_Aug05_part2.csv"
#OUT_PARQUET  = "interconnection_projects.parquet"

# ----------------------------------------------------------------------
# 2) HELPER: fetch details for ONE project -----------------------------
# ----------------------------------------------------------------------
def fetch_project(url: str) -> Dict[str, str]:
    """Return the metadata dict for a single interconnection project page."""
    html = requests.get(url, timeout=15).text
    soup = BeautifulSoup(html, "html.parser")

    script_tag = soup.find("script", id="__NEXT_DATA__")
    if script_tag is None:                          # fallback: scrape table
        rows = soup.select("table tr")
        return {
            r.find_all("td")[0].get_text(strip=True): 
            r.find_all("td")[1].get_text(" ", strip=True)
            for r in rows if r.find_all("td")
        }

    payload = json.loads(script_tag.string)
    return payload["props"]["pageProps"]["serializedProjectProps"]["json"]


In [30]:
df = pd.read_excel(INPUT_XLSX)
df[LINK_COLUMN] = df[LINK_COLUMN].apply(lambda x: x.split("|")[1] if isinstance(x, str) and "|" in x else None)
links = (df[LINK_COLUMN].dropna().unique())

records: List[Dict[str, str]] = []
errors  : List[str]           = []

for link in tqdm(links, desc="Scraping projects"):
    try:
        data = fetch_project(link)
        data["source_url"] = link          # keep a trace
        records.append(data)
    except Exception:
        errors.append(link)
        traceback.print_exc()
    time.sleep(PAUSE_SEC)

# assemble dataframe
df = pd.DataFrame(records)
print(f"\n✓ scraped {len(df)} projects "
        f"(failed: {len(errors)})")

# save
df.to_csv(OUT_CSV, index=False)
#df.to_parquet(OUT_PARQUET, index=False)

if errors:
    pathlib.Path("failed_links.txt").write_text("\n".join(errors))
    print("⚠ some links failed → see failed_links.txt")

Scraping projects:   0%|          | 0/10000 [00:00<?, ?it/s]Traceback (most recent call last):
  File "/var/folders/pv/386_wnhs3kn06cp1zjbsczv00000gq/T/ipykernel_70167/475812210.py", line 10, in <module>
    data = fetch_project(link)
  File "/var/folders/pv/386_wnhs3kn06cp1zjbsczv00000gq/T/ipykernel_70167/2261927793.py", line 38, in fetch_project
    return payload["props"]["pageProps"]["serializedProjectProps"]["json"]
KeyError: 'serializedProjectProps'
Scraping projects:  18%|█▊        | 1804/10000 [16:46<1:16:41,  1.78it/s]Traceback (most recent call last):
  File "/Users/tonyliu/Documents/DriftNet/data_centers/driftnet/lib/python3.9/site-packages/urllib3/connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
  File "/Users/tonyliu/Documents/DriftNet/data_centers/driftnet/lib/python3.9/site-packages/urllib3/connection.py", line 516, in getresponse
    httplib_response = super().getresponse()
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Py


✓ scraped 9998 projects (failed: 2)
⚠ some links failed → see failed_links.txt
